In [2]:
import torch
import torch.nn as nn
from torch.nn import functional as F
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)
block_size = 32
batch_size = 64
max_iters = 500
#eval_interval = 2500
learning_rate = 3e-3
eval_iters = 50
dropout = 0.2 #arunca noduri random in network pentru a evita overfilling
n_embed = 128
n_layer = 3
n_head = 4
dropout = 0.1


cuda


In [3]:
chars = ""
with open("brother_karamazov.txt", 'r', encoding='utf-8') as f:
        text = f.read()
        chars = sorted(list(set(text)))
        
vocab_size = len(chars)

In [4]:
string_to_int = {ch:i for i,ch in enumerate(chars)}
int_to_string = {i:ch for i,ch in enumerate(chars)}
encode = lambda s: [string_to_int[c] for c in s]
decode = lambda l: ''.join([int_to_string[i] for i in l])

encoded_txt = encode('hello')
decoded = decode(encoded_txt)

data = torch.tensor(encode(text), dtype=torch.long)
print(data[:100])

tensor([97,  0,  0,  0,  0,  0,  0, 41, 58, 55,  2, 23, 68, 65, 70, 58, 55, 68,
        69,  2, 32, 51, 68, 51, 63, 51, 76, 65, 72,  1,  1, 41, 68, 51, 64, 69,
        62, 51, 70, 55, 54,  2, 56, 68, 65, 63,  2, 70, 58, 55,  2, 39, 71, 69,
        69, 59, 51, 64,  2, 65, 56,  1,  1, 27, 75, 65, 54, 65, 68,  2, 25, 65,
        69, 70, 65, 75, 55, 72, 69, 61, 75,  1,  1, 52, 75,  2, 24, 65, 64, 69,
        70, 51, 64, 53, 55,  2, 28, 51, 68, 64])


In [5]:
n = int(0.8*len(data))
train_data = data[:n]
val_data = data[n:]

def get_batch(split):
    data  = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    #print(ix)
    x=torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y
x, y = get_batch('train')
print('input=')
print(x)
print('target')
print(y)

input=
tensor([[ 2, 70, 51,  ..., 70,  2, 70],
        [70, 58, 59,  ..., 51, 69, 69],
        [58, 55,  1,  ..., 58, 55,  2],
        ...,
        [ 2, 73, 59,  ..., 54,  2, 54],
        [70,  1, 59,  ..., 69, 69,  2],
        [54,  2, 68,  ..., 72, 55, 64]], device='cuda:0')
target
tensor([[70, 51, 61,  ...,  2, 70, 58],
        [58, 59, 69,  ..., 69, 69, 51],
        [55,  1, 95,  ..., 55,  2, 69],
        ...,
        [73, 59, 62,  ...,  2, 54, 51],
        [ 1, 59, 64,  ..., 69,  2, 51],
        [ 2, 68, 51,  ..., 55, 64, 57]], device='cuda:0')


In [6]:
@torch.no_grad()# tot ce se intampla aici sa nu ia seama la gradienti
def estimate_loss():
    out = {}
    model.eval() # trecem in modul de evaluare
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters): #ia un batch trece prin model, retine pierderile
            x, y = get_batch(split)
            logits, loss = model.forward(x,y)
            losses[k] = loss.item()
        out[split] = losses.mean() # media pierderilor
    model.train()# pune modelul in modul de antrenare
    return out

In [7]:
x = train_data[:block_size]
y = train_data[1:block_size+1]
for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print("when input is ", context, "target is", target)

when input is  tensor([97]) target is tensor(0)
when input is  tensor([97,  0]) target is tensor(0)
when input is  tensor([97,  0,  0]) target is tensor(0)
when input is  tensor([97,  0,  0,  0]) target is tensor(0)
when input is  tensor([97,  0,  0,  0,  0]) target is tensor(0)
when input is  tensor([97,  0,  0,  0,  0,  0]) target is tensor(0)
when input is  tensor([97,  0,  0,  0,  0,  0,  0]) target is tensor(41)
when input is  tensor([97,  0,  0,  0,  0,  0,  0, 41]) target is tensor(58)
when input is  tensor([97,  0,  0,  0,  0,  0,  0, 41, 58]) target is tensor(55)
when input is  tensor([97,  0,  0,  0,  0,  0,  0, 41, 58, 55]) target is tensor(2)
when input is  tensor([97,  0,  0,  0,  0,  0,  0, 41, 58, 55,  2]) target is tensor(23)
when input is  tensor([97,  0,  0,  0,  0,  0,  0, 41, 58, 55,  2, 23]) target is tensor(68)
when input is  tensor([97,  0,  0,  0,  0,  0,  0, 41, 58, 55,  2, 23, 68]) target is tensor(65)
when input is  tensor([97,  0,  0,  0,  0,  0,  0, 41, 58,

In [8]:
class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embed, head_size, bias = False) # transforma n_embed to head_size
        self.query = nn.Linear(n_embed, head_size, bias = False)
        self.value = nn.Linear(n_embed, head_size, bias = False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size))) 
        self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        B,T,C = x.shape
        k = self.key(x)
        q = self.query(x)
        wei = q @ k.transpose(-2,-1) * k.shape[-1]**-0.5
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf')) # ne asiguram c a programul nu triseaza cand se uita inainte
        wei = F.softmax(wei, dim = -1) # transformam in probabilitati
        wei = self.dropout(wei)
        v = self.value(x)
        out = wei @ v
        return out

class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)]) # 4 heads in paralel
        self.proj = nn.Linear(head_size * num_heads, n_embed) 
        self.dropout = nn.Dropout(dropout)
    def forward(self,x):
        out = torch.cat([h(x) for h in self.heads], dim = -1) #concatenam fiecare head -> (B, T, F)
        out = self.dropout(self.proj(out))
        return out

class FeedForward(nn.Module):
     def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(n_embed, 4 * n_embd), nn.ReLU(), nn.Linear(4 * n_embd, n_embd), nn.Dropout(dropout))
     def forward(self,x):
        return self.net(x)

class Block(nn.Module):
     def __init__(self, n_embed, n_head):
         super().__init__()
         head_size = n_embed // n_head
         self.sa = MultiHeadAttention(n_head, head_size)
         self.ffwd = FeedForward(n_embed)
         self.ln1 = nn.LayerNorm(n_embed)
         self.ln2 = nn.LayerNorm(n_embed)


     def forward(self,x):
         y = self.sa(x)
         x = self.ln1(x+y)
         y = self.ffwd(x)
         x = self.ln2(x+y)
         return x

class GPTLanguageModel(nn.Module):
     def __init__(self, vocab_size):
         super().__init__()
         self.token_embedding_table = nn.Embedding(vocab_size, n_embed)
         self.position_embedding_table = nn.Embedding(block_size, n_embed)
         #se initializeaza un tabel unde pt fiecare element din vocab se retin toate probabilitatile
         # ca urmatorul element sa fie alta litera
         self.blocks = nn.Sequential(*[Block(n_embed, n_head) for _ in range(n_layer)]) 
         self.in_f = nn.LayerNorm(n_embed) # final layer norm
         self.in_head = nn.Linear(n_embed, vocab_size)
         self.apply(self.__init__weights)


     def __init__weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean = 0.0, std = 0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean = 0.0, std = 0.02)
        
     def forward(self, index, targets=None):
         B, T = index.shape
         tok_emb = self.token_embedding_table(index)
         pos_emb = self.position_embedding_table(torch.arange(T, device = device)) # un lookup table
         x = tok_emb + pos_emb
         x = self.blocks(x)
         x = self.in_f(x)
         logits = self.in_head(x)
         if targets is None:
             loss = None
         else:
             B, T, C = logits.shape#Batch, Time(lungimea secventei), Channels(vocab_size)
             logits = logits.view(B*T, C)
             targets = targets.view(B*T) #se faec sa se potriveasca pentru inmultire
             loss = F.cross_entropy(logits, targets)# cat de gresit s-a ghicit
         return logits, loss

     def generate(self, index, max_new_tokens, temperature=0.8, top_k=40):
        for _ in range(max_new_tokens):
            index_cond = index[:, -block_size:]
            logits, _ = self.forward(index_cond)
            logits = logits[:, -1, :] / temperature
            
            # Top-k sampling (better quality than pure sampling)
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = float('-inf')
            
            probs = F.softmax(logits, dim=-1)
            index_next = torch.multinomial(probs, num_samples=1)
            index = torch.cat((index, index_next), dim=1)
        return index
model = GPTLanguageModel(vocab_size)
m = model.to(device)
context = torch.zeros((1,1), dtype=torch.long, device = device)
generated_chars = decode(m.generate(context, max_new_tokens=500)[0].tolist())
print(generated_chars)

	﻿	K.9ê‐;h46P31UtR(Vfôz2H7DZ ,—[M[tÆeNyv.Sa8ôMêàAG3Sx9Æ[]Oë?uïVZTubï88ï
H)8?8îY ;ü65:o6p6 RTPARV?êê5éêTb—V;MlôY3ê)pNG—!H1N5AGYàïCVN‘cê(TnAX?[ôDJFI)rOxhô‐H:‐nthcY	ïC2ë‐U]oà.M;yu3H;êïP1Y	l!Y﻿_!ïbvGRêGU3HJ	x?]RQUnt;AN‘MîVyu	lttp?ê‘éD﻿_H‘oëY	xnM
g;fR.œïnc[êG îHï—FxÆD—!GUN—[ôMe‘x]püa;n;KVS5m:a8Scn!ôFcçôV5V5Yq]TN‘Aîïa[üôAwê(hEh2yuGy)—dFZAoZ8q(ooYYctN‐aaJ)y6Q;Juu[o!.HIcî!ôQéP6‐nV]Jo;êGKD8oOTC3yunNe]êGëuaDxübTHnyFê?!22z)—huYYàï:3qNeîVyn4ZpM8Géx?;é—ïa]3ZZPP!1]pT‐8:RA[oYG:c4Wm[3.8Dm3﻿æ:c!)AFüoNôçDFb ;JcvàV


In [9]:
optimizer = torch.optim.AdamW(model.parameters(),  lr = learning_rate)
for iter in range(max_iters):
    if iter % eval_iters == 0:
        losses = estimate_loss()
        print(f'step:{iter}, train loss:{losses['train']:.2f}, val loss : {losses['val']:.2f}') 
    xb, yb = get_batch('train')
    logits, loss = model.forward(xb, yb)#se face predictia si compara cu yb ca sa calculeze cat de gresit a fost
    optimizer.zero_grad(set_to_none=True)
    loss.backward()#se calculeaza gradientul astfel incat sa vedem cum loss ul sa scada
    optimizer.step()#actualizam weights
print(loss.item())

step:0, train loss:4.63, val loss : 4.64
step:50, train loss:2.44, val loss : 2.44
step:100, train loss:2.30, val loss : 2.30
step:150, train loss:2.17, val loss : 2.17
step:200, train loss:2.08, val loss : 2.08
step:250, train loss:1.99, val loss : 2.02
step:300, train loss:1.94, val loss : 1.96
step:350, train loss:1.92, val loss : 1.91
step:400, train loss:1.86, val loss : 1.87
step:450, train loss:1.85, val loss : 1.85
1.8633673191070557


In [ ]:
context = torch.zeros((1,1), dtype=torch.long, device=device)
generated_chars = decode(m.generate(context, max_new_tokens=500)[0].tolist())
print(generated_chars)